# Diabetes Model Comparison and Training
This notebook compares all models, displays their metrics, selects the best one, and saves it as `diabetes_model.pkl`.

In [12]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

In [13]:
RANDOM_STATE = 42
DATA_PATH = "diabetes.csv"
print("Dataset Preview")
df_preview=pd.read_csv(DATA_PATH)
display(df_preview.head())
print(df_preview.describe())

Dataset Preview


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002   
min       0.000000    0.000000       0.000000       0.000000    0.000000   
25%       1.000000   99.000000      62.000000       0.000000    0.000000   
50%       3.000000  117.000000      72.000000      23.000000   30.500000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    31.992578                  0.471876   33.240885    0.348958  
std      7.884160                  0.331329   11.760232    0.476951  
min      0.000000                  

In [14]:
MODEL_OUT = "diabetes_model.pkl"

# Columns where a 0 is not physiologically possible -> treat as missing
ZERO_AS_MISSING_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

In [15]:
def load_and_clean_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    for col in ZERO_AS_MISSING_COLS:
        # Mark biologically-impossible zeros as missing, THEN compute the
        # fill value only from the valid (non-zero) entries. Computing the
        # median/mean while zeros are still present (as the original
        # notebook did) underestimates the fill value.
        df[col] = df[col].replace(0, np.nan)
        fill_value = df[col].median()
        df[col] = df[col].fillna(fill_value)

    return df


In [16]:

def build_candidates():
    """Return a dict of {name: (pipeline, param_grid)} to try."""
    candidates = {}

    candidates["logreg"] = (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
        ]),
        {"clf__C": [0.01, 0.1, 1, 10, 100]},
    )

    candidates["random_forest"] = (
        Pipeline([
            ("scaler", StandardScaler()),  # not required for trees, but harmless & keeps pipeline uniform
            ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
        {
            "clf__n_estimators": [200, 400],
            "clf__max_depth": [4, 6, 8, None],
            "clf__min_samples_leaf": [1, 2, 4],
        },
    )

    candidates["gradient_boosting"] = (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
        ]),
        {
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [2, 3, 4],
            "clf__learning_rate": [0.01, 0.05, 0.1],
        },
    )

    
    candidates["xgboost"] = (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", XGBClassifier(
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                verbosity=0,
            )),
        ]),
        {
            "clf__n_estimators": [100, 200, 300],
            "clf__max_depth": [2, 3, 4],
            "clf__learning_rate": [0.01, 0.05, 0.1],
            "clf__subsample": [0.8, 1.0],
        },
    )

    return candidates

In [17]:
df = load_and_clean_data(DATA_PATH)

x = df.drop(columns=["Outcome"])
y = df["Outcome"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

candidates = build_candidates()

best_name, best_score, best_estimator = None, -1, None
results = []

print("=" * 70)
print("Tuning candidate models with 5-fold stratified CV (scoring=ROC AUC)")
print("=" * 70)

for name, (pipeline, param_grid) in candidates.items():
    search = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
    )
    search.fit(x_train, y_train)

    cv_acc = cross_val_score(search.best_estimator_, x_train, y_train, cv=cv, scoring="accuracy").mean()

    print(f"\n[{name}]")
    print(f"  Best params      : {search.best_params_}")
    print(f"  CV ROC AUC       : {search.best_score_:.4f}")
    print(f"  CV Accuracy      : {cv_acc:.4f}")

    results.append((name, search.best_score_, cv_acc, search.best_estimator_))

    if search.best_score_ > best_score:
        best_name, best_score, best_estimator = name, search.best_score_, search.best_estimator_

print("\n" + "=" * 70)
print(f"Best model on CV: {best_name} (ROC AUC = {best_score:.4f})")
print("=" * 70)

# Final evaluation on held-out test set
y_pred = best_estimator.predict(x_test)
y_proba = best_estimator.predict_proba(x_test)[:, 1]

print(f"\nFinal Test Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Final Test ROC AUC  : {roc_auc_score(y_test, y_proba):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Refit best estimator on ALL data (train+test) for the deployed model
best_estimator.fit(x, y)
joblib.dump(best_estimator, MODEL_OUT)
print(f"\nSaved best pipeline (scaler + model) to '{MODEL_OUT}'")


Tuning candidate models with 5-fold stratified CV (scoring=ROC AUC)

[logreg]
  Best params      : {'clf__C': 0.1}
  CV ROC AUC       : 0.8452
  CV Accuracy      : 0.7655

[random_forest]
  Best params      : {'clf__max_depth': 4, 'clf__min_samples_leaf': 4, 'clf__n_estimators': 400}
  CV ROC AUC       : 0.8395
  CV Accuracy      : 0.7639

[gradient_boosting]
  Best params      : {'clf__learning_rate': 0.05, 'clf__max_depth': 2, 'clf__n_estimators': 100}
  CV ROC AUC       : 0.8289
  CV Accuracy      : 0.7703

[xgboost]
  Best params      : {'clf__learning_rate': 0.05, 'clf__max_depth': 2, 'clf__n_estimators': 100, 'clf__subsample': 0.8}
  CV ROC AUC       : 0.8387
  CV Accuracy      : 0.7687

Best model on CV: logreg (ROC AUC = 0.8452)

Final Test Accuracy : 0.7078
Final Test ROC AUC  : 0.8104

Confusion Matrix:
[[73 27]
 [18 36]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.73      0.76       100
           1       0.57